In [1]:
from enum import Enum

In [2]:
class Need(str, Enum):
    nutrition = "Nutrition"
    shelter_and_living_conditions = "Shelter and living conditions"
    hygiene = "Hygiene"
    clothing = "Clothing"
    healthcare = "Healthcare"
    education = "Education"
    communication_and_information = "Communication and information"
    mobility = "Mobility"


class Resource(str, Enum):
    freshwater = "Freshwater"
    marine_resources = "Marine Resources"
    wetlands = "Wetlands"
    metals_and_ores = "Metals and Ores"
    non_metallic_minerals = "Non-Metallic Minerals"
    fossil_fuels = "Fossil Fuels"
    agricultural_land = "Agricultural Land"
    forests = "Forests"
    urban_land = "Urban Land"
    biomass = "Biomass"


class Wellbeing(str, Enum):
    housing = "Housing"
    jobs = "Jobs"
    education = "Education"
    civic_engagement = "Civic Engagement"
    life_satisfaction = "Life Satisfaction"
    work_life_balance = "Work-Life Balance"
    income = "Income"
    community = "Community"
    environment = "Environment"
    health = "Health"
    safety = "Safety"


class Justice(str, Enum):
    distributional = "Distributional"
    procedural = "Procedural"
    corrective = "Corrective"
    recognitional = "Recognitional"
    transitional = "Transitional"


class PlanetaryBoundary(str, Enum):
    land_system_change = "Land-System Change"
    climate_change = "Climate Change"
    biosphere_integrity = "Biosphere Integrity"
    biogeochemical_flows = "Biogeochemical Flows"
    ocean_acidification = "Ocean Acidification"
    freshwater_use = "Freshwater Use"
    atmospheric_aerosol_loading = "Atmospheric Aerosol Loading"
    ozone_depletion = "Ozone Depletion"
    introduction_of_novel_entities = "Introduction of Novel Entities"


In [3]:
from pydantic import BaseModel, Field
from typing import Literal

In [4]:
def to_literal(enum_cls: Enum) -> type:
    return Literal[*[e.value for e in enum_cls]]

In [5]:
class TaxonomyExtraction(BaseModel):
    #thinking: str = Field(description="The model's reasoning process for extracting the taxonomy.")
    human_needs: list[to_literal(Need)]
    natural_resources: list[to_literal(Resource)]
    wellbeing: list[to_literal(Wellbeing)]
    justice: list[to_literal(Justice)]
    planetary_boundaries: list[to_literal(PlanetaryBoundary)]

In [6]:
import asyncio
import os
import time
from dotenv import load_dotenv

from google import genai
from openai import AsyncOpenAI
import pandas as pd
from tqdm.notebook import tqdm

from pprint import pprint

load_dotenv()

True

In [7]:
client = genai.Client()
#client = AsyncOpenAI(
#    base_url=os.getenv("GENERATION_API_URL"),
#    api_key=os.getenv("SCW_SECRET_KEY"),
#)

In [8]:
df = pd.read_parquet('../../results_557k/sample_2000_extracted_policies.parquet', columns=['openalex_id', 'chunk_idx', 'text'])
df

,openalex_id,chunk_idx,text
0,W4306385914,0,explore a new and effective method for mariti...
1,W3173622727,6,. Compositional analysis of lignocellulosic fe...
2,W4238379039,1,"2012), shown in overview, however the\n384 pre..."
3,W4408679953,0,affecting breastfeeding practice among mother...
4,W3092401747,1,for all ﬁve local sites in both August and No...
...,...,...,...
1995,W4379144753,0,study aimed to develop a combustion technolog...
1996,W4389428043,2,the context can be changed only to a limited ...
1997,W4313209439,0,a distinct signature of contemporary global w...
1998,W2408034890,0,Valplast® and DuraFlex™ are thermoplastic mat...


In [31]:
async def extract_taxonomy(
    text: str, prompt: str, model_name: str, client: genai.Client = client
) -> TaxonomyExtraction:
    try:
        def _call():
            return client.models.generate_content(
                model=model_name,
                contents=text,
                config=genai.types.GenerateContentConfig(
                    system_instruction=prompt.strip(),
                    thinking_config=genai.types.ThinkingConfig(thinking_level="minimal"),
                    temperature=0,
                    max_output_tokens=512,
                    response_mime_type="application/json",
                    response_schema=TaxonomyExtraction,
                ),
            )
        response = await asyncio.to_thread(_call)
        return TaxonomyExtraction.model_validate_json(response.text)
    except Exception as e:
        print('Error')
        if 'response' in locals():
            print(response.text)
        return f"Error: {e}"


async def batch_extract_taxonomy(
    texts: list[str], prompt: str, model_name: str, client: genai.Client = client
) -> list[TaxonomyExtraction]:
    results = await asyncio.gather(*[extract_taxonomy(text, prompt, model_name, client) for text in texts])
    return results


In [50]:
async def extract_taxonomy(
    text: str, prompt: str, model_name: str, client: AsyncOpenAI = client
) -> TaxonomyExtraction:
    try:
        response = await client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": prompt.strip()},
                {"role": "user", "content": text},
            ],
            temperature=0,
            max_tokens=512,
            response_format= {
                    "type": "json_schema",
                    "json_schema": {
                        "name": TaxonomyExtraction.__name__,
                        "schema": TaxonomyExtraction.model_json_schema(),
                    },
                },
            timeout=60,
            stream=False,
        )
        return TaxonomyExtraction.model_validate_json(response.choices[0].message.content)
    except Exception as e:
        print('Error')
        # show response if defined
        if 'response' in locals():
            print(response.choices[0].message.content)
        return f"Error: {e}"


async def batch_extract_taxonomy(
    texts: list[str], prompt: str, model_name: str, client: AsyncOpenAI = client
) -> list[TaxonomyExtraction]:
    results = await asyncio.gather(*[extract_taxonomy(text, prompt, model_name, client) for text in texts])
    return results

In [10]:
model_name = "gemini-3-flash-preview"  #"gemini-3.1-flash-lite-preview"
#model_name = "mistral-small-3.2-24b-instruct-2506"
#model_name = "qwen3-235b-a22b-instruct-2507"

In [23]:
system_prompt = """
You are a scientific literature tagger. Your task is to annotate an excerpt from a scientific article using a controlled taxonomy.

## TAXONOMY

### Human Needs (direct human needs addressed or impacted)
- Nutrition: food security, hunger, dietary intake, malnutrition, food access
- Shelter and living conditions: housing, settlement, living standards, built environment
- Hygiene: sanitation, clean water access, waste management, disease prevention practices
- Clothing: textiles, apparel, thermal comfort from garments
- Healthcare: medical services, health systems, disease treatment, mental health services
- Education: schooling, literacy, training, knowledge access
- Communication and information: internet access, media, information systems, connectivity
- Mobility: transportation, migration, physical displacement, access to movement

### Natural Resources (resources explicitly studied, used, or impacted)
- Freshwater: rivers, lakes, groundwater, drinking water, irrigation water
- Marine Resources: fish stocks, seafood, ocean-based resources, coastal ecosystems
- Wetlands: marshes, swamps, peatlands, mangroves
- Metals and Ores: iron, copper, lithium, rare earth elements, mining products
- Non-Metallic Minerals: sand, gravel, cement, phosphate, potash
- Fossil Fuels: coal, oil, natural gas, petroleum products
- Agricultural Land: cropland, farmland, pasture, arable land
- Forests: timber, wood, forest cover, deforestation, afforestation
- Urban Land: cities, urban sprawl, built-up areas, land use in urban contexts
- Biomass: organic matter, bioenergy crops, plant and animal matter used as resource

### Wellbeing Dimensions (aspects of human wellbeing explicitly discussed)
- Housing: access to adequate shelter, housing affordability, homelessness
- Jobs: employment, unemployment, labor market, livelihoods, decent work
- Education: access to schooling, educational outcomes, skill development
- Civic Engagement: political participation, democracy, voting, public involvement
- Life Satisfaction: subjective wellbeing, happiness, quality of life perception
- Work-Life Balance: leisure time, overwork, family-work conflict
- Income: wages, poverty, economic inequality, purchasing power
- Community: social cohesion, social capital, belonging, neighborhood ties
- Environment: access to green spaces, exposure to pollution, environmental quality of life
- Health: physical and mental health outcomes, morbidity, mortality, wellbeing
- Safety: crime, violence, disaster risk, personal security

### Justice Considerations (types of justice explicitly addressed)
- Distributional: fair allocation of benefits, burdens, costs and resources across groups or regions
- Procedural: fairness of decision-making processes, participation, representation, transparency
- Corrective: compensation, remediation, accountability for past wrongs or damages
- Recognitional: acknowledgment and respect of marginalized or vulnerable groups, identities, and cultures
- Transitional: justice in the context of societal transitions (e.g., energy transition, post-conflict), managing winners and losers of change

### Planetary Boundaries (Earth system boundaries explicitly discussed)
- Land-System Change: deforestation, land use change, habitat conversion at global scale
- Climate Change: greenhouse gas emissions, global warming, CO2 concentration, climate tipping points
- Biosphere Integrity: biodiversity loss, species extinction, ecosystem functioning
- Biogeochemical Flows: nitrogen and phosphorus cycles, nutrient pollution, fertilizer runoff
- Ocean Acidification: CO2 absorption by oceans, pH decrease, impacts on marine calcifiers
- Freshwater Use: global freshwater consumption, water scarcity, hydrological cycle disruption
- Atmospheric Aerosol Loading: particulate matter, air pollution, regional climate effects from aerosols
- Ozone Depletion: stratospheric ozone, UV radiation, ozone-depleting substances
- Introduction of Novel Entities: synthetic chemicals, plastics, microplastics, heavy metals, new substances with unknown Earth system impacts

## TAGGING RULES
1. **Be conservative**: only tag what is explicitly and substantively discussed, not background context or passing mentions.
2. **Do not infer**: if a concept is merely implied or loosely related, do not tag it.
3. **Prefer recall over precision**: a false negative (missed tag) is worse than a false positive (irrelevant tag).
4. **Typical output**: 0-2 tags per category. More than 3 tags in a single category should be rare and well-justified.
5. **Each tag may be used at most once** per category.
6. **Only relevant categories**: if a category is not relevant to the text, leave it empty.

Return only the JSON object.
"""


In [35]:
i = 1366
pprint(df.iloc[i].text)

('ial end members (EM4+EM5) was used as an indicator for the intensity of '
 'aeolian transport. From 1910 to 1930 and from 1980 to 2000 (Fig. 6), the '
 'proportion of aeolian material varied considerably because of intense dust '
 'storms. Those two periods were influenced strongly by aeolian conditions, '
 'and coincide with periods of strong aeolian transport at Chaiwopu Lake (Fig. '
 '6) (Ma et al.,\n'
 '2013), which were also dated to the intervals 1910-1930 and 1980-2000. '
 'Apparent synchrony of dust storms in western China is best explained by '
 'regional forcing mech- anisms. Regional climate was generally dry, but '
 'experi- enced strong oscillations from ca. 1910 to the 1930s in agreement '
 'with Chaiwopu Lake region inferred from orFig. 5. Modelled end-members of '
 'the terrigenous sediment fraction from Sayram Lake. a) Particle size '
 'distribution of EM1 and EM2. b) Particle size distribution of EM4 and EM5. '
 'c) Particle size distribution of EM3 with dustfall sa

In [36]:
res = await extract_taxonomy(df.text.iloc[i], system_prompt, model_name, client)
res

TaxonomyExtraction(human_needs=[], natural_resources=['Agricultural Land', 'Forests'], wellbeing=['Environment'], justice=[], planetary_boundaries=['Land-System Change', 'Climate Change', 'Atmospheric Aerosol Loading'])

In [37]:
# process by batch
batch_size = 20
rpm_quota = 1000
wait_time = 60 / (rpm_quota / batch_size)
results = []
for i in tqdm(range(0, len(df), batch_size)):
    batch = df.iloc[i:i+batch_size]
    texts = batch.text.values.tolist()
    preds = await batch_extract_taxonomy(texts, system_prompt, model_name, client)
    results.extend(preds)
    time.sleep(wait_time)

  0%|          | 0/100 [00:00<?, ?it/s]

In [41]:
tax = pd.DataFrame([r.model_dump() for r in results])
tax

,human_needs,natural_resources,wellbeing,justice,planetary_boundaries
0,"[Communication and information, Mobility]",[],[],[],[]
1,[Shelter and living conditions],"[Biomass, Non-Metallic Minerals]",[Housing],[],[]
2,[],"[Freshwater, Marine Resources]",[Environment],[],"[Biosphere Integrity, Biogeochemical Flows, In..."
3,"[Nutrition, Healthcare]",[],"[Health, Community]",[],[]
4,"[Shelter and living conditions, Healthcare]","[Freshwater, Agricultural Land]","[Housing, Health]",[],"[Climate Change, Freshwater Use]"
...,...,...,...,...,...
1995,[],[Fossil Fuels],[Environment],[],"[Climate Change, Atmospheric Aerosol Loading]"
1996,[Education],[],"[Jobs, Life Satisfaction, Community, Education]",[],[]
1997,[],[Marine Resources],[Environment],[],"[Climate Change, Biosphere Integrity]"
1998,[Healthcare],[],[Health],[],[Introduction of Novel Entities]


In [44]:
final = pd.concat([df, tax], axis=1)

In [50]:
embs = pd.read_parquet('../../results_557k/sample_2000_cleaned_chunks_embedding_Qwen3_06B.parquet')
embs.head()

,openalex_id,chunk_idx,text,embeddings
0,W4306385914,0,Based on the meteorological parameters drawn f...,"[-0.010414, 0.05417, -0.01009, 0.04498, -0.019..."
1,W3173622727,6,Compositional analysis of lignocellulosic feed...,"[-0.04724, -0.01901, -0.00936, 0.005135, -0.01..."
2,W4238379039,1,Bolivina striatula is an opportunist character...,"[-0.04166, 0.03516, -0.007732, 0.0777, 0.03888..."
3,W4408679953,0,NICU hospitalization can be a significant barr...,"[-0.03662, -0.02034, -0.00832, 0.104, 0.01773,..."
4,W3092401747,1,Downscaling three ESMs using a regional climat...,"[-0.0791, 0.04633, -0.00942, 0.04837, 0.00913,..."


In [53]:
final['embedding'] = embs.embeddings

In [55]:
final.to_parquet('../../results_557k/sample_2000_impact_taxonomy_gemini3_flash.parquet')

In [58]:
final.columns

Index(['openalex_id', 'chunk_idx', 'text', 'human_needs', 'natural_resources',
       'wellbeing', 'justice', 'planetary_boundaries', 'embedding'],
      dtype='str')

In [60]:
for col in ['human_needs', 'natural_resources', 'wellbeing', 'justice', 'planetary_boundaries']:
    print(final[col].explode().value_counts())
    print('---')

human_needs
Nutrition                        498
Healthcare                       497
Education                        339
Shelter and living conditions    260
Mobility                         201
Hygiene                          163
Communication and information    112
Clothing                          14
Name: count, dtype: int64
---
natural_resources
Agricultural Land        452
Freshwater               395
Biomass                  302
Forests                  206
Urban Land               182
Fossil Fuels             176
Marine Resources         153
Metals and Ores           79
Non-Metallic Minerals     67
Wetlands                  58
Name: count, dtype: int64
---
wellbeing
Health               755
Environment          668
Income               389
Jobs                 324
Education            271
Safety               217
Community            180
Life Satisfaction    155
Housing              142
Work-Life Balance     34
Civic Engagement      32
Name: count, dtype: int64
---
justice
D